# Session 7 · **Meet LlamaIndex**
## When a vector search isn't the right question to ask

---

## 0.1 · Where we are

Six sessions in, here's what we've built:

| Session | We learned |
|---|---|
| 3 | How vector search narrows a city down to one person |
| 4 | How to make that work for a hundred million people |
| 5 | How to build a RAG pipeline by hand — and how to check it isn't lying |
| 6 | How LangChain removes the plumbing |

And every single time, the skeleton was identical:

```
ingest → chunk → embed → store → retrieve → build prompt → answer
```

Notice what the "retrieve" step always meant: **find the passages most similar to the question.** Every variant we built — Memory, Branched, Adaptive, Corrective — changed *how we asked* or *what we did afterwards*. None of them changed what retrieval fundamentally was.

Today we question that.

---

## 0.2 · Three questions vector search can't answer well

Picture our coffee library — 15 pages, 8 sections each.

**Question 1:** *"What is the cold brew ratio?"*
Vector search is perfect here. One passage has the answer, and we find it.

**Question 2:** *"What do all these pages cover, overall?"*
Now think about what vector search does. It finds the **top 5 most similar passages**. But "overall" isn't in any passage — the answer requires *all 120 pieces*, not the closest 5. Top-k retrieval structurally cannot answer this. Ask for more passages and you just blow the context window.

**Question 3:** *"In the ashwagandha page, what does the Method section say?"*
We know exactly where the answer is. Why are we running a similarity search at all? We should be *navigating*, not searching.

> **The pattern:** a vector index is one *view* of your data — similarity. Some questions want a different view: a summary of everything, an exact keyword lookup, or a walk down a hierarchy.

**LlamaIndex is the framework that gives you those other views.**

---

## 0.3 · LangChain vs LlamaIndex

They overlap, and both can do most things. The difference is emphasis:

| | LangChain | LlamaIndex |
|---|---|---|
| Focus | **Orchestration** — how pieces flow | **Knowledge** — how data is structured |
| Core question | "What happens next?" | "What's the right view of this data?" |
| Its unit | the Runnable (`.invoke()`) | the **Node** (chunk + metadata) |
| Gives you | retrievers, chains, memory, agents | index types, query engines, routers |

**The analogy from your notes:** LangChain is the plumbing — it decides how water flows. LlamaIndex is the tank and filters — it decides how water is stored and purified.

Or, since we've been living in it: a vector DB alone is a playlist sorted only by *vibe*. LlamaIndex lets you also sort by exact lyrics, by artist summary, or by album hierarchy. LangChain then decides how those results flow into a conversation.

> **In real projects, people use both.** LlamaIndex manages the knowledge; LangChain manages the workflow. We'll do exactly that at the end.

---

## 0.4 · Today's plan

| Part | Topic |
|---|---|
| 1 | Setup, and the `Settings` object |
| 2 | **Document → Node** — the unit of knowledge |
| 3 | The five-line RAG that shows what a framework can hide |
| 4 | The four index types, built from the same nodes |
| 5 | Query engines, and the three knobs that matter |
| 6 | Retrievers vs query engines, and post-processing |
| 7 | Prompt templates — putting our grounding rules back |
| 8 | Routing: letting the LLM pick the index |
| 9 | Sub-questions: breaking a hard query into easy ones |
| 10 | Using a real vector store (Pinecone) |
| 11 | LlamaIndex + LangChain together |
| 12 | What we learned, and the honest trade-offs |

### 📘 About the code
Same rule as Sessions 5 and 6: **`for` loops, `if` statements, lists, dictionaries, f-strings.** No lambdas, no comprehensions, no classes.

### ⚠️ Version note
LlamaIndex renamed its Google packages. Two imports you'll find in older tutorials are **discontinued**:

| Old (dead) | Current |
|---|---|
| `llama_index.llms.gemini` → `Gemini` | `llama_index.llms.google_genai` → `GoogleGenAI` |
| `llama_index.embeddings.gemini` | `llama_index.embeddings.google_genai` → `GoogleGenAIEmbedding` |


---
# PART 1 · Setup and `Settings`

LlamaIndex has one idea LangChain doesn't: a **global `Settings` object**. You tell it once which LLM, which embedding model, and how to chunk — and every index and query engine you build afterwards picks those up automatically.

That's convenient and it's a trap. Convenient because you don't pass `llm=` into twenty constructors. A trap because when something behaves oddly, the cause may be a setting you configured three cells ago.

In [ ]:
!pip -q install llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-vector-stores-pinecone pinecone beautifulsoup4

import os
import glob

os.environ["GOOGLE_API_KEY"] = open("gemini_key2.txt").read().strip()
os.environ["PINECONE_API_KEY"] = open("api_key.txt").read().strip()

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core.node_parser import SentenceSplitter

Settings.llm = GoogleGenAI(model="gemini-2.0-flash", temperature=0)
Settings.embed_model = GoogleGenAIEmbedding(model_name="models/text-embedding-004")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

print("LLM       :", Settings.llm.model)
print("Embeddings:", Settings.embed_model.model_name)
print("Chunking  : size", Settings.node_parser.chunk_size,
      "overlap", Settings.node_parser.chunk_overlap)

### The three settings, and what each controls

| Setting | Controls | Why it matters |
|---|---|---|
| `Settings.llm` | who writes the final answer | `temperature=0` keeps it to the facts in your context |
| `Settings.embed_model` | how text becomes numbers | must be the same model for documents *and* questions |
| `Settings.node_parser` | how documents get cut up | too big → vectors average out; too small → sentences lose context |

Nothing new — these are Session 3, 5 and 6 ideas. Only the place you set them has changed.

---
# PART 2 · Document → Node

This is LlamaIndex's central abstraction, and it's worth slowing down for.

- A **Document** is a whole source: one PDF, one web page, one file.
- A **Node** is a piece of that document, carrying its text, its metadata, an id, and its relationships to neighbouring nodes.

Your notes call nodes *flashcards made out of a big book*, which is exactly right.

> **Why "Node" and not "chunk"?** A chunk is just text. A node knows where it came from, what's before and after it, and what document it belongs to. That's what makes hierarchy possible in Part 4 — you can't build a tree out of loose chunks.

We'll build nodes from our coffee pages, splitting by section as we did in Session 6.

In [ ]:
from bs4 import BeautifulSoup
from llama_index.core import Document


def read_one_page(filename):
    html = open(filename, encoding="utf-8", errors="ignore").read()
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "head"]):
        tag.decompose()

    title = soup.find("h1").get_text(strip=True)
    page_id = os.path.basename(filename).replace(".html", "")

    documents = []
    for heading in soup.find_all("h2"):
        section_name = heading.get_text(strip=True)

        section_text = ""
        for item in heading.next_siblings:
            if getattr(item, "name", None) == "h2":
                break
            if hasattr(item, "get_text"):
                section_text = section_text + " " + item.get_text(" ", strip=True)
        section_text = " ".join(section_text.split())

        if len(section_text) > 40:
            info = {"page": page_id, "title": title, "section": section_name}
            full_text = title + " - " + section_name + ". " + section_text
            documents.append(Document(text=full_text, metadata=info))

    return documents


documents = []
for filename in sorted(glob.glob("coffee_pages/*.html")):
    for document in read_one_page(filename):
        documents.append(document)

print("Documents:", len(documents))
print("First one :", documents[0].metadata)

### Turning Documents into Nodes

`SentenceSplitter` does the cutting. Our sections are already small, so most become a single node — but run it anyway and look at what a node carries.

In [ ]:
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
nodes = splitter.get_nodes_from_documents(documents)

print("Documents:", len(documents), "-> Nodes:", len(nodes))

first_node = nodes[0]
print("\nid       :", first_node.node_id[:12])
print("metadata :", first_node.metadata)
print("text     :", first_node.get_content()[:90], "...")
print("\nrelationships this node knows about:")
for kind in first_node.relationships:
    print("   ", kind)

> **Look at that relationships list.** The node knows which document it came from, and which nodes sit before and after it. That's the difference between a node and a chunk, and it's what Part 4's TreeIndex is built on.

> **Metadata goes into the embedding.** By default LlamaIndex includes metadata in the text that gets embedded — so `section: Method` actually helps a search for "how do I make it". You can control this with `excluded_embed_metadata_keys` when metadata is noise rather than signal.


---
# PART 3 · The five-line RAG

Here is all of Session 5, in five lines. Run it, then we'll talk about what just happened.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=3)

answer = query_engine.query("Is turmeric coffee caffeine free?")
print(answer)

### That was too easy, and that's the problem

In Session 5 that took us forty lines: embed the question, query Pinecone, unpack results, format context, fill a prompt, call the model, extract text.

Here it's two objects. But **every decision we made by hand is still being made** — just by someone else's defaults:

| Decision | Session 5 | Here |
|---|---|---|
| how many passages | our `how_many` argument | `similarity_top_k=3` |
| what the prompt says | we wrote every word | a built-in template |
| how answers combine | we pasted passages together | a "response mode" we didn't choose |
| what if nothing matches | our "say you don't know" rule | **no such rule** |

That last row should worry you. Let's see what the model was actually told.

In [ ]:
prompts = query_engine.get_prompts()

for name in prompts:
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(prompts[name].get_template()[:600])
    print()

> **This is the most important habit in this notebook.** Any framework that writes prompts for you is making decisions on your behalf. `get_prompts()` shows you exactly what's being sent. Read it before you trust it.
>
> Notice what the default template does *not* say: nothing about refusing when the context is thin, and nothing about citations. We'll put both back in Part 7.


### Seeing which nodes were used

The answer object carries its sources. This is the grounding check from Session 5 — did the answer come from the documents we think it did?

In [ ]:
answer = query_engine.query("How do I make masala coffee?")

print("ANSWER:", answer)
print("\nBuilt from these nodes:")
for source in answer.source_nodes:
    print("   score", round(source.score, 3), "|",
          source.node.metadata["page"][:32], "|", source.node.metadata["section"])

---
# PART 4 · The four index types

This is the heart of the session. **The same nodes, four different views.**

| Index | How it finds things | Best for | Cost |
|---|---|---|---|
| **VectorStoreIndex** | embedding similarity | "what drink is calming?" | one embedding per node |
| **SummaryIndex** | keeps every node, summarises across them | "what do these cover overall?" | reads everything at query time |
| **KeywordTableIndex** | a keyword → node lookup table | exact terms, codes, names | one LLM call per node to extract keywords |
| **TreeIndex** | a hierarchy built bottom-up | drill-down navigation | LLM calls to build each level |

> **The key insight:** you're not choosing a *database*. You're choosing a **retrieval strategy**. The nodes are the same in all four.

Let's build three of them from the same node list and ask each the same two questions.

In [ ]:
from llama_index.core import SummaryIndex, KeywordTableIndex

vector_index = VectorStoreIndex(nodes)
summary_index = SummaryIndex(nodes)

print("VectorStoreIndex built - embeds every node")
print("SummaryIndex built     - keeps every node, no embedding needed")

### The question vector search cannot answer

🎲 **Before you run it:** *"What kinds of Indian coffee do these pages cover overall?"*

The vector engine will fetch its 3 most similar passages. The summary engine will read **all 120 nodes**. Which one can actually answer "overall"?

In [ ]:
broad_question = "What kinds of Indian coffee do these pages cover overall?"

print("=== VectorStoreIndex (top 3 passages) ===")
vector_answer = vector_index.as_query_engine(similarity_top_k=3).query(broad_question)
print(vector_answer)
print("\nsaw", len(vector_answer.source_nodes), "nodes out of", len(nodes))

print("\n\n=== SummaryIndex (every node) ===")
summary_engine = summary_index.as_query_engine(response_mode="tree_summarize")
summary_answer = summary_engine.query(broad_question)
print(summary_answer)
print("\nsaw", len(summary_answer.source_nodes), "nodes out of", len(nodes))

### And the question where summary is the wrong tool

Now flip it. Ask something narrow and specific.

🎲 **Before you run it:** which will be faster, and which will be cheaper?

In [ ]:
import time

narrow_question = "What is the cold brew steeping time?"

start = time.time()
fast_answer = vector_index.as_query_engine(similarity_top_k=3).query(narrow_question)
vector_time = time.time() - start

start = time.time()
slow_answer = summary_engine.query(narrow_question)
summary_time = time.time() - start

print("VECTOR  :", round(vector_time, 1), "seconds |", str(fast_answer)[:110])
print("\nSUMMARY :", round(summary_time, 1), "seconds |", str(slow_answer)[:110])
print("\nSummaryIndex read every node to answer a question that lived in one of them.")

> **The lesson, and it's the session's whole point:** there is no best index. SummaryIndex answered the broad question that vector search structurally couldn't, and wasted time and tokens on the narrow one. *Choosing the right view is the skill.*

### `response_mode` — how answers get assembled

Notice we passed `response_mode="tree_summarize"` above. That's LlamaIndex's answer to "what do I do when there's more context than fits in one prompt":

| Mode | What it does | Use when |
|---|---|---|
| `compact` (default) | stuffs as much context as fits into one call | normal Q&A |
| `refine` | answers with node 1, then revises with node 2, and so on | accuracy over many nodes |
| `tree_summarize` | summarises in pairs, then summarises the summaries | broad "overall" questions |
| `no_text` | retrieves only, generates nothing | debugging your retrieval |

`no_text` is genuinely useful — it lets you check retrieval quality without paying for generation.

In [ ]:
check_engine = vector_index.as_query_engine(similarity_top_k=4, response_mode="no_text")
result = check_engine.query("caffeine content of ashwagandha coffee")

print("No answer generated - just what would have been used:\n")
for source in result.source_nodes:
    print("  ", round(source.score, 3), source.node.metadata["page"][:32],
          "|", source.node.metadata["section"])

### KeywordTableIndex — and an honest warning

A keyword index catches exact terms vector search glosses over. But building it costs **one LLM call per node** to extract keywords — with 120 nodes that's slow and not free.

We'll build it on a small slice so you can see the shape without burning your quota.

In [ ]:
small_set = nodes[:12]

keyword_index = KeywordTableIndex(small_set)
keyword_engine = keyword_index.as_query_engine()

print("Keyword index built from", len(small_set), "nodes")
print("\nSome extracted keywords:")

table = keyword_index.index_struct.table
shown = 0
for keyword in table:
    print("   ", keyword, "->", len(table[keyword]), "node(s)")
    shown = shown + 1
    if shown >= 12:
        break

> **TreeIndex** works the same way — `TreeIndex(nodes)` — building summary layers bottom-up so queries can drill down. It also costs LLM calls per level, so we describe rather than build it here. Reach for it with genuinely hierarchical documents: a textbook, a legal code, a long manual with chapters.


---
# PART 5 · Query engines and the three knobs

`as_query_engine()` hides a whole pipeline: retrieve → post-process → synthesise. Three arguments control most of its behaviour.

| Knob | Does | Session it comes from |
|---|---|---|
| `similarity_top_k` | how many nodes to fetch | Session 4's `k` |
| `response_mode` | how they're combined into an answer | new today |
| `node_postprocessors` | filtering and reranking after retrieval | Session 4's reranking |

🎲 **Before you run it:** raising `similarity_top_k` from 2 to 8 — better answer, worse answer, or just more expensive?

In [ ]:
question = "What spices are used across these coffees?"

for k_value in [2, 8]:
    engine = vector_index.as_query_engine(similarity_top_k=k_value)
    result = engine.query(question)
    print("top_k =", k_value, "| nodes used:", len(result.source_nodes))
    print("   ", str(result)[:190])
    print()

---
# PART 6 · Retrievers vs query engines

A distinction worth being precise about:

- A **retriever** finds nodes. No LLM, no answer. Just: question in, nodes out.
- A **query engine** = retriever + post-processing + an LLM that writes the answer.

Splitting them apart lets you test retrieval on its own — which is the fastest way to debug a bad answer. *If the right node never came back, no prompt will save you.*

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever

retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=5)
found_nodes = retriever.retrieve("is turmeric coffee caffeine free")

print("Retriever returns nodes, not an answer:\n")
for item in found_nodes:
    print("  ", round(item.score, 3), item.node.metadata["page"][:32],
          "|", item.node.metadata["section"])

### Post-processing: filtering weak nodes

`SimilarityPostprocessor` drops anything below a score you choose. This is Session 5's "evidence check" — if nothing scores well, better to send nothing than to send noise.

In [ ]:
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine

weak_filter = SimilarityPostprocessor(similarity_cutoff=0.5)

custom_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[weak_filter]
)

print("Without filter:", len(retriever.retrieve("quantum physics")), "nodes")

filtered = weak_filter.postprocess_nodes(retriever.retrieve("quantum physics"))
print("With filter   :", len(filtered), "nodes")
print("\nAn off-topic question should keep very few nodes - or none.")

---
# PART 7 · Prompt templates — putting our rules back

Remember Part 3: the default template says nothing about refusing, and nothing about citations. In Session 5 those were the two rules that stopped the Concierge inventing cafés.

Let's put them back.

In [ ]:
from llama_index.core import PromptTemplate

grounded_template = PromptTemplate(
    "You are the DilSeVector Concierge, a friendly coffee assistant.\n"
    "\n"
    "CONTEXT:\n"
    "{context_str}\n"
    "\n"
    "RULES:\n"
    "- Answer ONLY from the CONTEXT above.\n"
    "- If the CONTEXT does not answer the question, say you do not know. Never guess.\n"
    "- Keep it under 80 words.\n"
    "\n"
    "QUESTION: {query_str}\n"
    "ANSWER: "
)

grounded_engine = vector_index.as_query_engine(similarity_top_k=3)
grounded_engine.update_prompts(
    {"response_synthesizer:text_qa_template": grounded_template}
)

print("=== a question our pages CAN answer ===")
print(grounded_engine.query("Is turmeric coffee caffeine free?"))

print("\n=== a question our pages CANNOT answer ===")
print(grounded_engine.query("Which cafe in Dadar serves the best masala coffee, and what is its address?"))

> **The two placeholders are fixed names:** `{context_str}` is where retrieved nodes get pasted, `{query_str}` is the question. LlamaIndex fills them. Everything else in that template is yours.
>
> Compare the two answers above. The second one should decline. That behaviour is not in LlamaIndex — **it's in the sentence you wrote.**


---
# PART 8 · Routing: letting the LLM choose the index

We now have two views of the same nodes: vector (specific) and summary (broad). Part 4 showed that picking wrong gives bad or expensive answers.

In Session 5 we wrote an `if/elif` router by hand. `RouterQueryEngine` does it with the LLM instead: you describe each engine, and it picks one.

> **The critical part is the `description`.** That text is the *only* thing the LLM sees when choosing. A vague description means bad routing. This is prompt engineering wearing a different hat.

In [ ]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_index.as_query_engine(similarity_top_k=3),
    name="specific_facts",
    description=(
        "Useful for SPECIFIC questions about one coffee drink: ingredients, "
        "method, caffeine content, ratios, steeping times, cautions."
    )
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_index.as_query_engine(response_mode="tree_summarize"),
    name="overall_summary",
    description=(
        "Useful for BROAD questions that need all the documents at once: "
        "what topics are covered overall, comparisons across many drinks, "
        "general themes across the whole collection."
    )
)

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[vector_tool, summary_tool],
    verbose=True
)

print("Router built with 2 tools")

### Watch it choose
🎲 **Before you run it:** which tool should each question get? Write your guess down first.

In [ ]:
questions = [
    "What is the cold brew steeping time?",
    "What themes do all of these coffee pages have in common?"
]

for question in questions:
    print("=" * 70)
    print("Q:", question)
    result = router.query(question)
    print("A:", str(result)[:220])
    print()

---
# PART 9 · Sub-questions: breaking a hard query into easy ones

Session 5 taught us that *"compare X with Y"* fails when retrieval only finds X. Our fix then was to raise `k` and hope.

`SubQuestionQueryEngine` does something smarter: the LLM **splits the question into sub-questions**, runs each against the right tool, then merges the answers.

> **The cost:** one LLM call to split, one retrieval + generation per sub-question, one to merge. A three-part comparison can cost five calls. Use it where comparison quality matters, not everywhere.

### ⚠️ A trap worth knowing
`SubQuestionQueryEngine.from_defaults()` tries to load an **OpenAI** question generator by default and crashes without an OpenAI key. The fix is one argument: pass `LLMQuestionGenerator`, which uses whatever is in `Settings.llm`.

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen import LLMQuestionGenerator

question_generator = LLMQuestionGenerator.from_defaults()   # uses Settings.llm, not OpenAI

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[vector_tool, summary_tool],
    question_gen=question_generator,
    use_async=False,
    verbose=True
)

result = sub_question_engine.query(
    "Compare the caffeine content of ashwagandha coffee and turmeric coffee, "
    "and say which suits someone avoiding caffeine."
)

print("\n" + "=" * 70)
print(result)

> **Look at the verbose output above.** You can see the sub-questions it generated and which tool each went to. That's Session 5's Branched RAG, decided by the model instead of by our `if/elif`.


---
# PART 10 · Using a real vector store

Everything so far kept vectors in memory — fine for 120 nodes, useless for a million. Swapping in Pinecone changes **two lines**, and nothing else in the notebook needs to know.

That's the same swappability lesson as Session 6, in a different framework.

In [ ]:
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
pinecone_index = pc.Index("coffee-hybrid")

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

UPLOAD = False

if UPLOAD:
    stored_index = VectorStoreIndex(nodes, storage_context=storage_context)
    print("Uploaded", len(nodes), "nodes to Pinecone")
else:
    stored_index = VectorStoreIndex.from_vector_store(vector_store)
    print("Connected to the existing Pinecone index")

pinecone_engine = stored_index.as_query_engine(similarity_top_k=3)
print("\n", pinecone_engine.query("What is in masala coffee?"))

> **`from_vector_store` is the pattern for production.** Index once, then every service connects to the same store without re-embedding anything. The index lives in the database, not in your notebook's memory.


---
# PART 11 · LlamaIndex + LangChain together

The two frameworks aren't rivals — they solve different halves of the problem. A common real setup:

```
LlamaIndex  →  manages knowledge: nodes, index types, routing, sub-questions
LangChain   →  manages the conversation: history, chains, streaming, tools
```

The join is simple: **use a LlamaIndex query engine as the retrieval step inside a LangChain chain.** Everything else in Session 6 stays as it was.

In [ ]:
!pip -q install langchain-core langchain-google-genai

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

lc_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

lc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the DilSeVector Concierge. Answer ONLY from the CONTEXT.\n"
     "If the CONTEXT does not answer it, say you do not know.\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{question}")
])

lc_chain = lc_prompt | lc_llm | StrOutputParser()


def get_context_from_llamaindex(question):
    """LlamaIndex does the knowing. It hands back text."""
    found = retriever.retrieve(question)
    lines = []
    for item in found:
        lines.append("[" + item.node.metadata["page"] + "] " + item.node.get_content())
    return "\n\n".join(lines)


def ask(question):
    context = get_context_from_llamaindex(question)      # LlamaIndex: retrieval
    return lc_chain.invoke({"context": context, "question": question})   # LangChain: the flow


print(ask("How do I make jaggery coffee?"))

> **Why bother combining them?** Because now you get LlamaIndex's routing and index types on the knowledge side, *and* LangChain's streaming, history and tool-calling on the conversation side. Neither framework does both halves as well alone.
>
> The bridge is just text: retrieval returns nodes, you format them into a string, the chain takes it from there. No special adapter needed.


---
# PART 12 · What we learned

### The four views, and when to reach for each

| Index | Reach for it when | Cost |
|---|---|---|
| **VectorStoreIndex** | specific questions, "what is / how do I" | cheap, one embedding per node |
| **SummaryIndex** | "what does this cover overall" | reads everything, every query |
| **KeywordTableIndex** | exact codes, names, terms | one LLM call per node to build |
| **TreeIndex** | genuine hierarchies you navigate | LLM calls per level to build |

### The honest trade-offs

**What LlamaIndex gave us:**
- Retrieval strategies beyond similarity — and *a name for the problem* that vector search alone can't answer broad questions
- Nodes that know their neighbours, which is what makes hierarchy possible
- Query engines that route and decompose without hand-written `if/elif`
- Backend swapping at the cost of two lines

**What it took away, and you must take back:**
- **It writes your prompts.** The default template has no refusal rule and no citation rule. We saw that in Part 3 and fixed it in Part 7.
- **`Settings` is global.** Convenient until a setting three cells back is the reason your answers changed.
- **Costs hide behind simple calls.** `KeywordTableIndex(nodes)` is one line and 120 LLM calls. `SubQuestionQueryEngine` can be five calls per question.
- **Still no grounding check.** Same as LangChain: no component verifies the answer is true. Your Session 5 checks remain the only thing doing that.

> *"Session 5 taught you what a RAG pipeline does. Session 6 taught you how a framework hides the plumbing. Today you learned that similarity search was never the only question to ask — and that a framework choosing your retrieval strategy is also a framework choosing your prompt. Read `get_prompts()` before you trust anything."*

---

## Homework

1. **Read before you trust.** Call `get_prompts()` on the router and the sub-question engine. Find the template that decides how sub-questions get generated.

2. **Prove the summary point.** Write one question that VectorStoreIndex genuinely cannot answer and SummaryIndex can, using our coffee pages. Bring it next session.

3. **Port your checks.** Take `find_citations` and `check_citations_real` from Session 5. Run them on `answer.source_nodes` from any engine here. They still work — that's the point.

4. **Cost the router.** Add `TokenCountingHandler` and measure how many tokens a routed question costs versus a plain vector query.

## Quick reference

| I want to… | Use |
|---|---|
| set the model / embeddings / chunking once | `Settings.llm`, `Settings.embed_model`, `Settings.node_parser` |
| make nodes from documents | `SentenceSplitter().get_nodes_from_documents(docs)` |
| semantic search | `VectorStoreIndex(nodes)` |
| answer "what is this all about" | `SummaryIndex(nodes)` + `response_mode="tree_summarize"` |
| exact keyword lookup | `KeywordTableIndex(nodes)` |
| get nodes without an answer | `index.as_retriever().retrieve(q)` or `response_mode="no_text"` |
| see the prompt being used | `engine.get_prompts()` |
| replace the prompt | `engine.update_prompts({"response_synthesizer:text_qa_template": tpl})` |
| drop weak matches | `SimilarityPostprocessor(similarity_cutoff=…)` |
| pick an index automatically | `RouterQueryEngine` + `QueryEngineTool` |
| split a hard question | `SubQuestionQueryEngine` + `LLMQuestionGenerator` |
| use a real database | `PineconeVectorStore` + `StorageContext` |
